# Lab 2 — Experiment tracking con MLflow

**Taller: MLOps en la práctica — del notebook a producción** · UNI
**Duración:** ~75 min · **Modalidad:** guiado + retos

## 🎯 Objetivos
1. Entender el problema que resuelve el *experiment tracking* (y por qué Excel no escala).
2. Registrar experimentos con **MLflow**: parámetros, métricas, artefactos y el modelo.
3. Comparar decenas de runs de forma sistemática y elegir un ganador **con criterio**.
4. Usar la UI de MLflow (local y en Colab).

## El problema real

Ayer guardamos `modelo_churn_v1.joblib`. Hoy tu jefa pregunta:
*"¿Probaste regresión logística? ¿Y con menos árboles? ¿Cuál fue el AUC de cada uno? ¿Con qué datos?"*

Sin tracking, la respuesta vive en tu memoria, en celdas sobreescritas o en un Excel llamado `experimentos_final_v3_AHORA_SI.xlsx`. **MLflow** registra cada entrenamiento automáticamente: qué código, qué parámetros, qué métricas, qué modelo.

In [ ]:
%pip install -q mlflow==3.15.2 scikit-learn==1.8.0 pandas pyarrow

In [ ]:
# === Datos: mismo flujo del Lab 1, condensado en funciones (así se ve un proyecto ordenado) ===
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

URL_DATOS = ""  # <-- URL raw de GitHub del instructor (si no tienes el CSV local)

FEATURES_NUM = ["edad", "meses_antiguedad", "cargo_mensual_soles", "gb_datos_mes",
                "minutos_llamadas_mes", "lineas_adicionales", "tickets_soporte_6m",
                "caidas_servicio_mes", "dias_ultimo_pago_vencido", "factura_electronica"]
FEATURES_CAT = ["departamento", "plan", "tipo_contrato"]
RANDOM_STATE = 42

def cargar_datos():
    if os.path.exists("churn_telco_peru.csv"):
        df = pd.read_csv("churn_telco_peru.csv")
    elif URL_DATOS:
        df = pd.read_csv(URL_DATOS); df.to_csv("churn_telco_peru.csv", index=False)
    else:
        raise FileNotFoundError("Sube churn_telco_peru.csv o define URL_DATOS")
    df = df.drop_duplicates(subset="id_cliente", keep="first")
    X = df[FEATURES_NUM + FEATURES_CAT]
    y = df["churn"]
    return train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

def crear_preprocesador():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), FEATURES_NUM),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), FEATURES_CAT),
    ])

X_train, X_test, y_train, y_test = cargar_datos()
print("train:", X_train.shape, "| test:", X_test.shape)

## 1. Configurar MLflow

MLflow necesita dos cosas:
- Un **tracking backend**: dónde guarda runs y métricas. Usaremos SQLite local (`mlflow.db`) — en una empresa sería un servidor compartido (o Databricks/AzureML, que hablan el mismo protocolo).
- Un **experimento**: la carpeta lógica que agrupa runs de un mismo problema.

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn-andestel")
print("Tracking URI:", mlflow.get_tracking_uri())

## 2. Tu primer run: anatomía de un experimento registrado

Un **run** = una ejecución de entrenamiento. Dentro de un run registramos:

| Qué | Función | Ejemplo |
|---|---|---|
| Parámetros | `mlflow.log_param(s)` | `n_estimators=300` |
| Métricas | `mlflow.log_metric(s)` | `roc_auc=0.84` |
| Artefactos | `mlflow.log_artifact` | gráfico de la matriz de confusión, config |
| Modelo | `mlflow.sklearn.log_model` | el pipeline completo |
| Tags | `mlflow.set_tag` | `autor=jordan`, `dataset=v2025` |

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score

def evaluar(modelo, X_test, y_test):
    proba = modelo.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    return {
        "roc_auc": roc_auc_score(y_test, proba),
        "f1": f1_score(y_test, pred),
        "recall": recall_score(y_test, pred),
        "precision": precision_score(y_test, pred),
    }

with mlflow.start_run(run_name="rf-baseline"):
    params = {"n_estimators": 300, "max_depth": 8, "class_weight": "balanced"}
    modelo = Pipeline([
        ("preproc", crear_preprocesador()),
        ("clf", RandomForestClassifier(**params, random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    modelo.fit(X_train, y_train)
    metricas = evaluar(modelo, X_test, y_test)

    mlflow.log_params(params)
    mlflow.log_param("modelo_tipo", "random_forest")
    mlflow.log_metrics(metricas)
    mlflow.set_tag("dataset", "churn_telco_peru_2025")
    # Nota: MLflow >=3.15 usa 'skops' por defecto (formato seguro pero estricto con pipelines complejos).
    # Para el taller usamos cloudpickle, el formato clásico compatible con cualquier pipeline de sklearn.
    mlflow.sklearn.log_model(modelo, name="modelo", serialization_format="cloudpickle")

    print({k: round(v, 4) for k, v in metricas.items()})

In [ ]:
# También podemos registrar artefactos: por ejemplo, la matriz de confusión como imagen
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

with mlflow.start_run(run_name="rf-baseline-con-grafico"):
    modelo.fit(X_train, y_train)
    mlflow.log_params(params)
    mlflow.log_metrics(evaluar(modelo, X_test, y_test))

    fig, ax = plt.subplots(figsize=(4, 4))
    ConfusionMatrixDisplay.from_estimator(modelo, X_test, y_test,
                                          display_labels=["se queda", "churn"], ax=ax, colorbar=False)
    fig.tight_layout()
    fig.savefig("matriz_confusion.png", dpi=120)
    plt.close(fig)
    mlflow.log_artifact("matriz_confusion.png")
    print("Run con artefacto registrado ✅")

## 3. El poder real: barrer 12 configuraciones sin perderse

Aquí es donde el tracking se paga solo. Entrenaremos **3 familias de modelos × 4 configuraciones** y MLflow guardará todo. Nota que el código de entrenamiento es el mismo — solo cambia el diccionario de configuración.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
import time

def construir_modelo(tipo, hp):
    if tipo == "logreg":
        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE, **hp)
    elif tipo == "random_forest":
        clf = RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **hp)
    elif tipo == "hist_gb":
        clf = HistGradientBoostingClassifier(random_state=RANDOM_STATE, **hp)
    return Pipeline([("preproc", crear_preprocesador()), ("clf", clf)])

EXPERIMENTOS = [
    ("logreg", {"C": 0.1, "class_weight": "balanced"}),
    ("logreg", {"C": 1.0, "class_weight": "balanced"}),
    ("logreg", {"C": 10.0, "class_weight": None}),
    ("random_forest", {"n_estimators": 100, "max_depth": 6, "class_weight": "balanced"}),
    ("random_forest", {"n_estimators": 300, "max_depth": 8, "class_weight": "balanced"}),
    ("random_forest", {"n_estimators": 500, "max_depth": 12, "class_weight": "balanced_subsample"}),
    ("random_forest", {"n_estimators": 300, "max_depth": None, "class_weight": None}),
    ("hist_gb", {"max_iter": 150, "learning_rate": 0.1, "max_depth": 4}),
    ("hist_gb", {"max_iter": 300, "learning_rate": 0.05, "max_depth": 6}),
    ("hist_gb", {"max_iter": 300, "learning_rate": 0.1, "max_depth": None}),
    ("hist_gb", {"max_iter": 500, "learning_rate": 0.03, "max_depth": 8}),
    ("hist_gb", {"max_iter": 150, "learning_rate": 0.3, "max_depth": 3}),
]

for i, (tipo, hp) in enumerate(EXPERIMENTOS, 1):
    with mlflow.start_run(run_name=f"{tipo}-{i:02d}"):
        t0 = time.time()
        m = construir_modelo(tipo, hp)
        m.fit(X_train, y_train)
        mets = evaluar(m, X_test, y_test)
        mlflow.log_param("modelo_tipo", tipo)
        mlflow.log_params(hp)
        mlflow.log_metrics(mets)
        mlflow.log_metric("segundos_entrenamiento", time.time() - t0)
        mlflow.set_tag("barrido", "sesion1")
        mlflow.sklearn.log_model(m, name="modelo", serialization_format="cloudpickle")
        print(f"{i:02d} {tipo:14s} AUC={mets['roc_auc']:.4f} recall={mets['recall']:.3f} ({time.time()-t0:.1f}s)")

## 4. Comparar y elegir con criterio

**Opción A — la UI de MLflow** (la verás en la demo del instructor y puedes abrirla tú):

- **Local:** en una terminal: `mlflow ui --backend-store-uri sqlite:///mlflow.db` → abre http://localhost:5000
- **Colab:** ejecuta la celda siguiente (usa el proxy de puertos de Colab).

**Opción B — por código** con `mlflow.search_runs`, ideal para reportes automáticos.

In [ ]:
# Solo Colab: levanta la UI de MLflow y genera un enlace proxy
# (en local, usa la terminal como se indica arriba y SALTA esta celda)
try:
    from google.colab import output  # noqa
    import subprocess, time as _t
    subprocess.Popen(["mlflow", "ui", "--backend-store-uri", "sqlite:///mlflow.db", "--port", "5000"])
    _t.sleep(6)
    print("Abre la UI aquí ↓")
    output.serve_kernel_port_as_window(5000)
except ImportError:
    print("No estás en Colab: levanta la UI con -> mlflow ui --backend-store-uri sqlite:///mlflow.db")

In [ ]:
# Comparación por código: tabla de líderes
runs = mlflow.search_runs(order_by=["metrics.roc_auc DESC"])
cols = ["tags.mlflow.runName", "params.modelo_tipo", "metrics.roc_auc",
        "metrics.recall", "metrics.f1", "metrics.segundos_entrenamiento"]
tabla = runs[[c for c in cols if c in runs.columns]].head(10).round(4)
tabla

### 🗣️ Discusión: ¿el mejor AUC gana?

Mira la tabla: ¿el modelo con mejor AUC es también el más rápido? ¿El más simple? En AndesTel, el modelo se ejecuta 1 vez al día por lote — la velocidad casi no importa, pero la **estabilidad y explicabilidad sí** (Indecopi puede pedir explicar por qué llamaste a un cliente).

**Criterio de selección propuesto** (negociado con el equipo, documentado):
1. `roc_auc` dentro del top (± 0.01 del mejor)
2. Entre esos, mayor `recall` (el negocio prioriza detectar fugas)
3. Empate → el más simple

In [ ]:
# Selección automática del campeón según el criterio acordado
mejor_auc = runs["metrics.roc_auc"].max()
candidatos = runs[runs["metrics.roc_auc"] >= mejor_auc - 0.01]
campeon = candidatos.sort_values("metrics.recall", ascending=False).iloc[0]
RUN_ID_CAMPEON = campeon["run_id"]
print("🏆 Campeón:", campeon["tags.mlflow.runName"],
      "| AUC:", round(campeon["metrics.roc_auc"], 4),
      "| recall:", round(campeon["metrics.recall"], 4))
print("run_id:", RUN_ID_CAMPEON)

# Guardamos el run_id para el Lab 3
with open("run_id_campeon.txt", "w") as f:
    f.write(RUN_ID_CAMPEON)

## 🎯 Retos (20 min)

**Reto 1 (todos):** agrega tu propia configuración al barrido (otra combinación de hiperparámetros o `GradientBoostingClassifier`) con `run_name="apellido-custom"`. ¿Superas al campeón?

**Reto 2 (todos):** registra como métrica adicional el **AUC en train**. Compárala con la de test en la tabla: ¿qué configuraciones muestran más sobreajuste?

**Reto 3 (avanzado):** usa `mlflow.sklearn.autolog()` y entrena un modelo más. Inspecciona en la UI qué registró automáticamente que nosotros no registrábamos a mano.

**Reto 4 (avanzado):** con `mlflow.search_runs`, genera un gráfico de dispersión AUC vs. tiempo de entrenamiento, coloreado por familia de modelo, y regístralo como artefacto en un run nuevo llamado `reporte-barrido`.

## 📌 Lo que te llevas

- ✅ Cada entrenamiento queda registrado: parámetros, métricas, artefactos, modelo y entorno.
- ✅ Comparar 12 (o 200) experimentos toma segundos, no memoria heroica.
- ✅ La selección del campeón es un **criterio explícito y auditable**, no "me pareció mejor".
- ✅ En tu empresa: levanta un tracking server compartido y todo el equipo ve los experimentos de todos.

**Problema pendiente:** el campeón sigue siendo "un run con un ID". ¿Cómo lo convertimos en *EL modelo oficial v3 aprobado para producción*? → **Lab 3: Model Registry**.